charger tous les modele + prédiction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FakeNewsProject'

In [ ]:
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/FakeNewsProject'

test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

In [ ]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/FakeNewsProject'

# charger données test
test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

# charger modèles déjà sauvegardés
lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

BASE = '/content/drive/MyDrive/FakeNewsProject'

# RECHARGEMENT OBLIGATOIRE
y_test = np.load(f'{BASE}/data/y_test.npy')

lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

BASE = '/content/drive/MyDrive/FakeNewsProject'

# charger données
test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

# RECRÉER TF-IDF (OBLIGATOIRE)
train = pd.read_csv(f'{BASE}/data/processed/train.csv')
X_train = train['clean_text'].fillna('')
y_train = train['label']

tfidf = TfidfVectorizer(max_features=100000, ngram_range=(1,2),
                        sublinear_tf=True, min_df=2)

X_tr = tfidf.fit_transform(X_train)

# RECRÉER LOGISTIC REGRESSION
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_tr, y_train)

In [ ]:
!pip -q install transformers torch shap lime

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FakeNewsProject'

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#  Vérifier et utiliser GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

# Charger données
test = pd.read_csv(f'{BASE}/data/processed/test.csv')
y_test = test['label'].values

# Charger BERT
tok_bert = AutoTokenizer.from_pretrained(f'{BASE}/models/bert_final')
mdl_bert = AutoModelForSequenceClassification.from_pretrained(
    f'{BASE}/models/bert_final'
).to(device).eval()

# Fonction prédiction
def bert_predict_proba(texts):
    """Retourne probabilité FAKE pour chaque texte."""
    enc = tok_bert(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors='pt'
    )

    #  envoyer les données au GPU
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = mdl_bert(**enc).logits

    #  ramener résultat vers CPU pour numpy
    return torch.softmax(logits, dim=-1)[:, 0].cpu().numpy()

# Texte de test
texts_test = test['clean_text'].fillna('').tolist()

# Prédictions par batch
bert_probs = []

for i in range(0, len(texts_test), 32):
    print(f"Batch {i}/{len(texts_test)}")  # suivi progression
    bert_probs.extend(bert_predict_proba(texts_test[i:i+32]))

bert_probs = np.array(bert_probs)

print(f"✓ BERT prédictions : {bert_probs.shape}")

In [ ]:
import os
print(os.listdir(f'{BASE}/preds'))

ensemble vote pondéré : LSTM + BERT +LR

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import joblib, pickle
import numpy as np

# Charger les prédictions LSTM et LR sauvegardées
#np.save(f'{BASE}/preds/bert_probs.npy', bert_probs)
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

# Supposons qu'on a déjà sauvegardé ces arrays
lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')

# Ensemble pondéré : BERT pèse plus car meilleur modèle
ensemble_probs = (0.6 * bert_probs +
                  0.25 * lstm_probs +
                  0.15 * lr_probs)

ensemble_preds = (ensemble_probs > 0.5).astype(int)

print("=== TABLEAU COMPARATIF FINAL ===")
models_results = {
    "TF-IDF + LR" : (lr_probs,   'baseline'),
    "BiLSTM+GloVe": (lstm_probs, 'deep'),
    "BERT"        : (bert_probs, 'transformer'),
    "Ensemble"    : (ensemble_probs, 'ensemble')
}
for name, (probs, _) in models_results.items():
    preds_ = (probs > 0.5).astype(int)
    print(f"{name:20s}  F1={f1_score(y_test,preds_,average='macro'):.4f}  "
          f"AUC={roc_auc_score(y_test,probs):.4f}")

LIME : explication article par article

In [ ]:
!pip install lime

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np

explainer_lime = LimeTextExplainer(class_names=['FAKE', 'REAL'])

#  modèle utilisé pour LIME (Logistic Regression)
def predict_for_lime(texts):
    X = tfidf.transform(texts)
    probs_real = lr.predict_proba(X)[:, 1]
    return np.column_stack([1 - probs_real, probs_real])

# choisir des erreurs du modèle ensemble
wrong_idx = np.where((ensemble_preds != y_test) & (y_test == 0))[0][:3]

for idx in wrong_idx:
    article = X_test.iloc[idx][:300]

    exp = explainer_lime.explain_instance(
        article,
        predict_for_lime,
        num_features=10,
        num_samples=200
    )

    print(f"\n--- Article #{idx} (FAKE mal classé REAL) ---")
    print(f"Texte : {article[:100]}...")
    print("Top features LIME :")

    for feat, weight in exp.as_list():
        direction = "→ FAKE" if weight > 0 else "→ REAL"
        print(f"{feat:20s} {weight:+.4f} {direction}")

# sauvegarde HTML
exp.save_to_file(f'{BASE}/figures/lime_explanation.html')